In [1]:
!pip install -q langchain==0.0.232

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
google-adk 1.15.1 requires pydantic<3.0.0,>=2.0, but you have pydantic 1.10.24 which is incompatible.
google-genai 1.39.1 requires pydantic<3.

In [2]:
!pip install -q openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.7 MB/s eta 0:00:00


In [4]:
import openai
from google.colab import userdata

openai.api_key = userdata.get('OPENAI_KEY')

In [5]:
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_KEY')

# Chains ใน LangChain

## เนื้อหา

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [6]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

### 1. LLMChain

- Chain แรกเป็น simple chain
- เป็นแค่ Prompt ต่อกับ LLM

In [7]:
from langchain.chains import LLMChain

In [8]:
llm = ChatOpenAI(temperature=0.9)

In [9]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that make {product}?"
)

In [10]:
chain = LLMChain(
    llm = llm,
    prompt = prompt
)

In [11]:
product = "Queen Size Sheet Set"
chain.run(product)

'"Regal Rest Beddings"'

### 2. SimpleSequentialChain

In [12]:
from langchain.chains import SimpleSequentialChain

In [13]:
llm = ChatOpenAI(temperature=0.9)
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that make {product}?"
)
chain1 = LLMChain(
    llm = llm,
    prompt = first_prompt
)

In [14]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words describtion for the following company : {company_name}"
)
chain2 = LLMChain(
    llm = llm,
    prompt = second_prompt
)

In [15]:
overall_simple_chain = SimpleSequentialChain(
    chains  = [chain1,chain2], # input -> chain1 -> chain2 -> output
    verbose = True
)

In [16]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
"Royal Comfort Linens"
"Royal Comfort Linens provides luxurious bedding and towels for a regal sleep experience. Elevate your home with our premium linens."

> Finished chain.


'"Royal Comfort Linens provides luxurious bedding and towels for a regal sleep experience. Elevate your home with our premium linens."'

In [17]:
# prompt template 3
third_prompt = ChatPromptTemplate.from_template(
    "List 10 thing that I can but frorm {company_description}"
)
chain3 = LLMChain(
    llm = llm,
    prompt = third_prompt
)

In [18]:
overall_simple_chain = SimpleSequentialChain(
    chains = [chain1,chain2,chain3],
    verbose = True
)

In [19]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
"Royal Dimensions Home"
"Royal Dimensions Home specializes in luxurious and elegant home design, offering top-quality furnishings and decor for a regal lifestyle."
1. Luxurious sofas and sectionals
2. Elegant dining room tables and chairs
3. Top-quality bedding sets and linens
4. Regal chandeliers and lighting fixtures
5. Stylish accent chairs and ottomans
6. Premium rugs and carpets
7. Exclusive art pieces and wall decor
8. High-end kitchen accessories and appliances
9. Custom-made window treatments and draperies
10. Designer furniture and home decor accessories

> Finished chain.


'1. Luxurious sofas and sectionals\n2. Elegant dining room tables and chairs\n3. Top-quality bedding sets and linens\n4. Regal chandeliers and lighting fixtures\n5. Stylish accent chairs and ottomans\n6. Premium rugs and carpets\n7. Exclusive art pieces and wall decor\n8. High-end kitchen accessories and appliances\n9. Custom-made window treatments and draperies\n10. Designer furniture and home decor accessories'

### 3.SequentialChain

###

In [20]:
from langchain.chains import SequentialChain

In [21]:
review = """
สินค้าส่งโคตรไว ประมาณ2วันถึงมือเลย ห่อมาหนาแน่นดีมากๆ น่าจะมาจากเกาหลี แต่เครื่องผลิตที่จีน หลังจากได้ของแล้วอย่าลืมลงทะเบียนรับประกัน 2 ปีกับไมโครซอฟด้วย \
เล่นเกมมาแล้ว 3 วัน ปกติดี ซื้อเกมง่ายกว่าที่คิด ไม่ติด oops เลยตัวเครื่องร้อนเวลาเล่นไปนานๆ แต่ไม่มาก ข้อเสียงอย่างเดียวคือติดตั้งเกมนานไปนิด
"""

In [22]:
llm = ChatOpenAI(temperature=0.9)

In [23]:
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1 : input=Review and output=English_review
chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
    output_key="English_Review"
    )

In [24]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    output_key="summary"
    )

In [25]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input = Review and output = language
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    output_key = "language"
    )

In [26]:
# prompt template 4 : follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following"
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output=followup_message
chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    output_key = "followup_message"
    )

In [27]:
# overall_chain: input = Review
# and output = English_Review, summary, followup_message
overall_chain = SequentialChain(
    chains = [chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review","summary","followup_message"],
    verbose = True
)

In [28]:
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'Review': '\nสินค้าส่งโคตรไว ประมาณ2วันถึงมือเลย ห่อมาหนาแน่นดีมากๆ น่าจะมาจากเกาหลี แต่เครื่องผลิตที่จีน หลังจากได้ของแล้วอย่าลืมลงทะเบียนรับประกัน 2 ปีกับไมโครซอฟด้วย เล่นเกมมาแล้ว 3 วัน ปกติดี ซื้อเกมง่ายกว่าที่คิด ไม่ติด oops เลยตัวเครื่องร้อนเวลาเล่นไปนานๆ แต่ไม่มาก ข้อเสียงอย่างเดียวคือติดตั้งเกมนานไปนิด\n',
 'English_Review': "The product arrived very quickly, about 2 days. The packaging was very well packed, it seems to come from Korea but manufactured in China. After receiving the product, don't forget to register for the 2-year warranty with Microsoft. I've been playing games for 3 days now and it's been working fine. Buying games is easier than I thought, no lagging so far. The only downside is that the game installation takes a bit long.",
 'summary': 'The reviewer received the product quickly, was impressed with the packaging, and has had a positive experience playing games on it so far, although the game installation process takes longer than expected.',
 'followup_messa

### 4.Router Chain

In [29]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

In [30]:
prompt_infos = [
    {
        "name":"physics",
        "description":"Good for answering questions about physics",
        "prompt_template":physics_template
    },
    {
        "name":"math",
        "description":"Good for answering math questions",
        "prompt_template":math_template
    },
    {
        "name":"History",
        "description":"Good for answering history quesitons",
        "prompt_template":history_template
    },
    {
        "name":"computer science",
        "description":"Good for answering computer science question",
        "prompt_template":computerscience_template
    }
]

In [31]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.prompts import PromptTemplate

In [32]:
llm = ChatOpenAI(temperature=0)

In [33]:
destination_chains = {}
for p_info in prompt_infos:
    print(p_info)
    name            = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template = prompt_template)
    chain = LLMChain(
        llm = llm,
        prompt = prompt
    )
    destination_chains[name] = chain

{'name': 'physics', 'description': 'Good for answering questions about physics', 'prompt_template': "You are a very smart physics professor. You are great at answering questions about physics in a conciseand easy to understand manner. When you don't know the answer to a question you admitthat you don't know.\n\nHere is a question:\n{input}"}
{'name': 'math', 'description': 'Good for answering math questions', 'prompt_template': 'You are a very good mathematician. You are great at answering math questions. You are so good because you are able to break down hard problems into their component parts,\nanswer the component parts, and then put them togetherto answer the broader question.\n\nHere is a question:\n{input}'}
{'name': 'History', 'description': 'Good for answering history quesitons', 'prompt_template': 'You are a very good historian. You have an excellent knowledge of and understanding of people,events and contexts from a range of historical periods. You have the ability to think,

In [34]:
destination = [f"{p['name']:18} : {p['description']}" for p in prompt_infos]
destination

['physics            : Good for answering questions about physics',
 'math               : Good for answering math questions',
 'History            : Good for answering history quesitons',
 'computer science   : Good for answering computer science question']

In [35]:
destinations_str = "\n".join(destination)
print(destinations_str)

physics            : Good for answering questions about physics
math               : Good for answering math questions
History            : Good for answering history quesitons
computer science   : Good for answering computer science question


In [36]:
destinations_str

'physics            : Good for answering questions about physics\nmath               : Good for answering math questions\nHistory            : Good for answering history quesitons\ncomputer science   : Good for answering computer science question'

In [37]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain  = LLMChain(
    llm = llm,
    prompt = default_prompt
)

In [38]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

<>:12: SyntaxWarning: invalid escape sequence '\ '
<>:12: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-2160379643.py:12: SyntaxWarning: invalid escape sequence '\ '
  "destination": string \ name of the prompt to use or "DEFAULT"


In [39]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations = destinations_str
)
router_prompt = PromptTemplate(
    template = router_template,
    input_variables = ["input"],
    output_parser = RouterOutputParser()
)
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [40]:
chain = MultiPromptChain(
    router_chain = router_chain,
    destination_chains = destination_chains,
    default_chain = default_chain,
    verbose = True
)

In [41]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...


/usr/local/lib/python3.12/dist-packages/langchain/chains/llm.py:275: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the electromagnetic radiation emitted by a perfect absorber of radiation, known as a black body. A black body absorbs all radiation that falls on it and emits radiation across the entire electromagnetic spectrum. The spectrum of black body radiation is continuous and depends only on the temperature of the black body. This phenomenon is described by Planck's law, which states that the intensity of radiation emitted by a black body at a given wavelength is proportional to the temperature of the body and the wavelength raised to the fifth power."

In [42]:
chain.run("whai is 2+2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2+2'}
> Finished chain.


'Thank you for the compliment! The answer to the question "what is 2+2" is 4.'

In [43]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}

ValueError: Received invalid destination chain name 'biology'